# Filter Data Points by UK County Boundaries

This notebook demonstrates how to filter latitude/longitude points from a DataFrame based on whether they fall within a specific UK county boundary.

## Requirements
```bash
pip install pandas geopandas shapely
```

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

## 1. Load UK Counties Shapefile

Download and load the UK Counties and Unitary Authorities shapefile from the official UK government data source.

In [ ]:
# URL to UK Counties shapefile (December 2024 boundaries)
shapefile_url = "https://ckan.publishing.service.gov.uk/dataset/7d535263-3fcc-4a14-ba37-d34f36381fa5/resource/7d535263-3fcc-4a14-ba37-d34f36381fa5/download/ctyua_dec_2024_boundaries_uk_buc.zip"

# Load the shapefile
print("Loading UK counties shapefile...")
counties_gdf = gpd.read_file(shapefile_url)

print(f"Loaded {len(counties_gdf)} counties/unitary authorities")
print(f"CRS: {counties_gdf.crs}")
print(f"\nColumns: {counties_gdf.columns.tolist()}")
counties_gdf.head()

## 2. Explore Available Counties

In [ ]:
# Find the column containing county names
# Typically 'CTYUA24NM' for 2024 data
county_name_column = 'CTYUA24NM'  # Adjust if needed

# List all available counties
print(f"Available counties ({len(counties_gdf)} total):\n")
for i, county in enumerate(sorted(counties_gdf[county_name_column].unique()), 1):
    print(f"{i:3d}. {county}")

## 3. Load Your CSV Data

Load your CSV file with columns: time, id, latitude, longitude, temperature

In [ ]:
# Option A: Load from your actual CSV file
# df = pd.read_csv('your_data.csv')

# Option B: Create example data for demonstration
df = pd.DataFrame({
    'time': pd.date_range('2024-01-01', periods=10, freq='h'),
    'id': range(1, 11),
    'latitude': [51.5074, 53.4808, 51.4545, 52.4862, 50.8198,  # London, Manchester, Kent, Birmingham, Brighton
                 53.8008, 51.2787, 52.6369, 53.9599, 54.9784],  # Liverpool, Southampton, Northampton, York, Newcastle
    'longitude': [-0.1278, -2.2426, 0.3456, -1.8904, -0.1358,
                  -2.9916, -1.3262, -1.1398, -1.0873, -1.6178],
    'temperature': [15.2, 12.8, 14.5, 13.9, 16.1, 11.5, 14.8, 13.2, 12.1, 10.9]
})

print(f"Loaded {len(df)} data points")
df.head()

## 4. Convert DataFrame to GeoDataFrame

Create Point geometries from latitude/longitude coordinates.

In [ ]:
# Create Point geometries (note: Point takes (longitude, latitude) order)
geometry = [Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])]

# Create GeoDataFrame
points_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Ensure both GeoDataFrames use the same CRS
points_gdf = points_gdf.to_crs(counties_gdf.crs)

print(f"Created GeoDataFrame with {len(points_gdf)} points")
print(f"Points CRS: {points_gdf.crs}")
points_gdf.head()

## 5. Filter Points by Selected County

Use spatial join to filter only the points that fall within the selected county boundary.

In [ ]:
# Select a specific county
selected_county_name = 'Kent'  # Change this to your desired county

# Get the county boundary
selected_county = counties_gdf[counties_gdf[county_name_column] == selected_county_name]

if selected_county.empty:
    print(f"Error: County '{selected_county_name}' not found!")
    print(f"Please choose from the list above.")
else:
    # Perform spatial join to filter points within the county
    # predicate='within' ensures points are inside the county boundary
    filtered_points = gpd.sjoin(points_gdf, selected_county, how='inner', predicate='within')
    
    print(f"\nOriginal data: {len(points_gdf)} points")
    print(f"Points in {selected_county_name}: {len(filtered_points)} points")
    
    # Display the filtered results
    if len(filtered_points) > 0:
        display_cols = ['time', 'id', 'latitude', 'longitude', 'temperature']
        print(f"\nFiltered data for {selected_county_name}:")
        filtered_points[display_cols]

## 6. Visualize the Results (Optional)

Plot the county boundary and the filtered points.

In [ ]:
if not selected_county.empty and len(filtered_points) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    
    # Plot county boundary
    selected_county.plot(ax=ax, color='lightblue', edgecolor='black', alpha=0.5)
    
    # Plot all points (gray)
    points_gdf.plot(ax=ax, color='gray', markersize=50, alpha=0.3, label='All points')
    
    # Plot filtered points (red)
    filtered_points.plot(ax=ax, color='red', markersize=100, label=f'Points in {selected_county_name}')
    
    ax.set_title(f'Data Points in {selected_county_name}', fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    plt.tight_layout()
    plt.show()
else:
    print("No points found in the selected county to visualize.")

## 7. Export Filtered Results

Save the filtered data back to a CSV file or convert to a regular pandas DataFrame.

In [ ]:
if not selected_county.empty and len(filtered_points) > 0:
    # Convert back to regular DataFrame (remove geometry column)
    result_df = pd.DataFrame(filtered_points.drop(columns='geometry'))
    
    # Clean up extra columns from the spatial join
    columns_to_keep = ['time', 'id', 'latitude', 'longitude', 'temperature']
    result_df = result_df[[col for col in columns_to_keep if col in result_df.columns]]
    
    print(f"Final filtered dataset:")
    print(result_df)
    
    # Save to CSV (uncomment to save)
    # output_filename = f'filtered_data_{selected_county_name.replace(" ", "_")}.csv'
    # result_df.to_csv(output_filename, index=False)
    # print(f"\nSaved to {output_filename}")

## 8. Reusable Function

Here's a compact function to filter points by county in one step.

In [ ]:
def filter_dataframe_by_county(df, county_name, counties_gdf, 
                               lat_col='latitude', lon_col='longitude',
                               county_col='CTYUA24NM'):
    """
    Filter DataFrame points that fall within a UK county boundary.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with latitude and longitude columns
    county_name : str
        Name of the county to filter by
    counties_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing county boundaries
    lat_col : str
        Name of latitude column (default: 'latitude')
    lon_col : str
        Name of longitude column (default: 'longitude')
    county_col : str
        Name of county name column in counties_gdf (default: 'CTYUA24NM')
    
    Returns:
    --------
    pandas.DataFrame
        Filtered DataFrame with only points in the selected county
    """
    # Create Point geometries
    geometry = [Point(lon, lat) for lon, lat in zip(df[lon_col], df[lat_col])]
    points_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
    
    # Match CRS
    points_gdf = points_gdf.to_crs(counties_gdf.crs)
    
    # Select county
    county_boundary = counties_gdf[counties_gdf[county_col] == county_name]
    
    if county_boundary.empty:
        raise ValueError(f"County '{county_name}' not found in shapefile")
    
    # Spatial join
    filtered = gpd.sjoin(points_gdf, county_boundary, how='inner', predicate='within')
    
    # Return as DataFrame, keeping only original columns
    return pd.DataFrame(filtered[df.columns])


# Example usage:
# filtered_df = filter_dataframe_by_county(df, 'Kent', counties_gdf)
# print(filtered_df)

## Quick Reference

### Complete workflow in a few lines:

```python
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load shapefile
counties = gpd.read_file('https://ckan.publishing.service.gov.uk/dataset/.../ctyua_dec_2024_boundaries_uk_buc.zip')

# Load your data
df = pd.read_csv('your_data.csv')

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, 
                       geometry=[Point(lon, lat) for lon, lat in zip(df['longitude'], df['latitude'])],
                       crs='EPSG:4326').to_crs(counties.crs)

# Filter by county
county = counties[counties['CTYUA24NM'] == 'Kent']
filtered = gpd.sjoin(gdf, county, how='inner', predicate='within')

# Convert back to DataFrame
result = pd.DataFrame(filtered.drop(columns='geometry'))
```